# 04 - Hyperparameter Tuning

Este notebook ajusta hiperparametros del modelo baseline y compara resultados con la version inicial.

Salida principal:
- Tabla baseline vs tuned
- Metricas guardadas en `../artifacts/metrics/model_comparison.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Carga de datos

DATA_PATH = Path("../data/train.csv")
df = pd.read_csv(DATA_PATH)

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt",
]
target = "SalePrice"

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
# Baseline model (mismo enfoque que en 03_modeling)

baseline_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline R2: {baseline_r2:.4f}")

In [ ]:
# Busqueda de hiperparametros (RandomizedSearchCV)

search_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "regressor",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

param_distributions = {
    "regressor__n_estimators": [100, 200, 300, 500],
    "regressor__max_depth": [None, 10, 20, 30],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
    "regressor__max_features": ["sqrt", "log2", None],
}

random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)

print("Best params:")
print(random_search.best_params_)
print(f"Best CV RMSE: {-random_search.best_score_:.2f}")

In [ ]:
# Evaluacion final modelo tuneado

best_model = random_search.best_estimator_
tuned_pred = best_model.predict(X_test)

tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))
tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_r2 = r2_score(y_test, tuned_pred)

print(f"Tuned RMSE: {tuned_rmse:.2f}")
print(f"Tuned MAE: {tuned_mae:.2f}")
print(f"Tuned R2: {tuned_r2:.4f}")

In [ ]:
# Comparativa baseline vs tuned

comparison = pd.DataFrame(
    [
        {
            "version": "baseline_random_forest",
            "rmse": round(baseline_rmse, 4),
            "mae": round(baseline_mae, 4),
            "r2": round(baseline_r2, 4),
        },
        {
            "version": "tuned_random_forest",
            "rmse": round(tuned_rmse, 4),
            "mae": round(tuned_mae, 4),
            "r2": round(tuned_r2, 4),
        },
    ]
)

comparison

In [ ]:
# Guardar comparativa

OUT_PATH = Path("../artifacts/metrics/model_comparison.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUT_PATH, index=False)

print(f"Comparativa guardada en: {OUT_PATH}")

## Interpretacion recomendada para el informe

- Si `tuned_rmse` < `baseline_rmse`, hubo mejora en precision.
- Si `tuned_mae` < `baseline_mae`, el error medio bajo.
- Si `tuned_r2` > `baseline_r2`, el modelo explica mejor la variabilidad del precio.

Usad la tabla `comparison` directamente en `doc.pdf` (apartado de ajuste de hiperparametros).